# **Q1. Data Loading**

**Understand the Concept**

- Use the Telco Customer Churn dataset for this tabular ANN example.
- `Churn` is the binary target.

In [ ]:
import pandas as pd

df = pd.read_csv("telco_customer_churn_4000.csv")
print(df.shape)
print(df.head())

## **Explanation**

- The CSV is loaded into a DataFrame.
- The target column is `Churn`.

# **Q2. Data Inspection**

**Understand the Concept**

- Inspect columns, data types, missing values, and target distribution before training.

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nChurn:")
print(df["Churn"].value_counts())

## **Explanation**

- Inspection tells us what preprocessing is required.
- Target counts show the class distribution.

# **Q3. Preprocessing**

**Understand the Concept**

- Neural networks need numerical inputs.
- Remove the customer ID because it is only an identifier.

In [ ]:
df = df.drop(columns=["customerID"])
print(df.columns.tolist())

## **Explanation**

- `customerID` is not a useful predictive feature, so it is removed.

# **Q4. Encode Features and Target**

**Understand the Concept**

- Convert `Churn` to 0/1.
- Convert categorical features using one-hot encoding.

In [ ]:
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
X = pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True)
y = df["Churn"]
print(X.shape, y.shape)

## **Explanation**

- The target becomes binary.
- Categorical features become numerical columns.

# **Q5. Feature Engineering**

**Understand the Concept**

- Feature engineering means creating or changing features to help a model.
- For this beginner ANN, the encoded features are sufficient.

In [ ]:
X = X.copy()
print("Number of features:", X.shape[1])

## **Explanation**

- No extra feature is required for this basic implementation.

# **Q6. Train/Test Split**

**Understand the Concept**

- Training data learns parameters.
- Test data is kept for final evaluation.
- Stratification helps preserve class proportions.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## **Explanation**

- 80% is used for training and 20% for testing.
- The test set is not used to update the model.

# **Q7. Scaling**

**Understand the Concept**

- Scaling puts features on comparable ranges.
- Fit the scaler only on training data.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(X_train_scaled.shape)

## **Explanation**

- The scaler learns values from training data only.
- The same transformation is applied to test data.

# **Q8. Tensor Conversion**

**Understand the Concept**

- PyTorch models use tensors.
- Binary targets are shaped as one output per sample.

In [ ]:
import torch

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1,1)
print(X_train_tensor.shape, y_train_tensor.shape)

## **Explanation**

- Features are `float32` tensors.
- Targets have shape `(samples, 1)`.

# **Q9. ANN Architecture**

**Understand the Concept**

- The input size equals the number of features.
- Hidden layers learn patterns.
- One output is used for binary classification.

In [ ]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)
print(model)

## **Explanation**

- The final layer produces one logit for churn classification.

# **Q10. Training**

**Understand the Concept**

- Training repeats forward pass, loss, backpropagation, and parameter update.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):
    logits = model(X_train_tensor)
    loss = loss_fn(logits, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Final training loss:", loss.item())

## **Explanation**

- Adam updates the ANN parameters.
- BCE with logits is suitable for binary classification.

# **Q11. Validation / Evaluation**

**Understand the Concept**

- Evaluation checks performance without updating parameters.

In [ ]:
model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(X_test_tensor))
    predictions = (probabilities >= 0.5).int()

print(predictions[:10].flatten())

## **Explanation**

- `no_grad()` disables gradient tracking.
- Sigmoid converts logits to probabilities.

# **Q12. Accuracy, Precision, Recall and F1**

**Understand the Concept**

- Accuracy measures overall correctness.
- Precision measures correctness of positive predictions.
- Recall measures how many actual positives were found.
- F1 balances precision and recall.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = y_test.values
y_pred = predictions.numpy().flatten()
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1-score :", f1_score(y_true, y_pred))

## **Explanation**

- These metrics give more detail than accuracy alone.
- Precision and recall are useful when the class distribution is not perfectly balanced.

# **Q13. Confusion Matrix**

**Understand the Concept**

- A confusion matrix shows correct and incorrect predictions for both classes.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
plt.imshow(cm)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("ANN Confusion Matrix")
plt.colorbar()
plt.show()
print(cm)

## **Explanation**

- The matrix shows true negatives, false positives, false negatives, and true positives.

# **Q14. ROC-AUC**

**Understand the Concept**

- ROC-AUC measures how well the model separates the two classes across thresholds.
- It uses prediction probabilities.

In [ ]:
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_true, probabilities.numpy().flatten())
print("ROC-AUC:", auc)

## **Explanation**

- A higher ROC-AUC generally indicates better class separation.

# **Q15. Logistic Regression vs ANN**

**Understand the Concept**

- Logistic Regression is a traditional ML baseline.
- Compare it with ANN on the same data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

print("Logistic Regression:", accuracy_score(y_test, lr_pred))
print("ANN:", accuracy_score(y_true, y_pred))

## **Explanation**

- Compare performance, complexity, training time, interpretability, dataset size, and resource requirements.
- More complexity does not automatically mean a better model.